# Train the square classifier (Colab)

**Pipeline step:** model training. All training logic lives in `scripts/train.py` in the repo; this notebook only moves files and runs it on Colab's GPU.

**Inputs** (upload once to Google Drive, folder `MyDrive/chess_vision/`):
- `crops.zip`: the dataset from `scripts/build_dataset.py` (`train/`, `val/`, `test/`, one folder per class)
- `train.py`: copy of `scripts/train.py`

**Outputs** (written to `MyDrive/chess_vision/outputs/`, so they survive the Colab session ending):
- `model.pt`: best epoch by validation accuracy
- `train_log.csv`: loss and accuracy per epoch
- test results: `confusion_matrix.png`, `test_mistakes.png`, `test_mistakes.csv`

**Before running:** Runtime > Change runtime type > **T4 GPU**.

## 1. Check the GPU
Should print a Tesla T4. If it says no GPU, change the runtime type and rerun.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Mount Google Drive
Gives this session access to `MyDrive`, where the inputs are and the outputs go.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/chess_vision'
!ls -lh {DRIVE}

## 3. Unzip the dataset to local disk
Reading 20k small files straight from Drive is slow; Colab's local disk is fast.

In [ ]:
!mkdir -p /content/data && unzip -q -o {DRIVE}/crops.zip -d /content/data
!for s in train val test; do echo $s $(find /content/data/crops/$s -name '*.jpg' | wc -l); done

## 4. Train
Fine-tunes all layers of pretrained ResNet18 for 10 epochs (Adam, lr 1e-4), light augmentation. Prints one line per epoch; `<- saved` marks a new best validation accuracy. About 5 min per epoch on a T4.

In [ ]:
!python {DRIVE}/train.py --data /content/data/crops --out {DRIVE}/outputs

## 5. Training curves
Train vs. validation loss and accuracy per epoch. A val curve that rises while train keeps falling means overfitting. Figure for the report.

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
log = pd.read_csv(f'{DRIVE}/outputs/train_log.csv')
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
log.plot(x='epoch', y=['train_loss', 'val_loss'], ax=ax[0], marker='o', title='Loss')
log.plot(x='epoch', y=['train_acc', 'val_acc'], ax=ax[1], marker='o', title='Accuracy')
plt.tight_layout(); plt.savefig(f'{DRIVE}/outputs/training_curves.png', dpi=150); plt.show()
log

## 6. Test: predict every test square
**Pipeline step:** evaluation. The test split (games test01 Byrne-Fischer 1956 and test02 Kasparov-Topalov 1999) was never used for training or for picking the epoch, so these are the honest numbers for the report. Validation accuracy is slightly optimistic because we chose the best epoch with it.

Loads the best `model.pt`, reuses `build_model` and `EVAL_TF` from `train.py` (same preprocessing as validation), and predicts all 5,440 test crops.

In [ ]:
import sys, numpy as np, torch
from pathlib import Path
from torch.utils.data import DataLoader
from torchvision import datasets
sys.path.insert(0, DRIVE)
from train import build_model, EVAL_TF

device = 'cuda' if torch.cuda.is_available() else 'cpu'
ckpt = torch.load(f'{DRIVE}/outputs/model.pt', map_location=device)
classes = ckpt['classes']
model = build_model(len(classes))
model.load_state_dict(ckpt['state_dict'])
model.to(device).eval()   # eval mode: BatchNorm uses its learned statistics, not the batch's

test_set = datasets.ImageFolder('/content/data/crops/test', EVAL_TF)
assert test_set.classes == classes, 'test folders must match the training classes'
preds = []
with torch.no_grad():   # no gradients needed: we only predict
    for images, _ in DataLoader(test_set, batch_size=128, num_workers=2):   # no shuffle, so order matches test_set.samples
        preds += model(images.to(device)).argmax(1).tolist()   # highest logit = predicted class
preds = np.array(preds)
labels = np.array([y for _, y in test_set.samples])
paths = [p for p, _ in test_set.samples]
print(len(preds), 'test squares predicted')

## 7. Per-square accuracy and per-class precision / recall / F1
Accuracy alone is misleading here: 64% of test squares are empty, so a model that always says "empty" already scores 64%. The per-class table shows whether the rare pieces (queens, kings) are recognized too.
- **Precision:** of the squares predicted as this class, how many really are.
- **Recall:** of the squares that really are this class, how many were found.
- **F1:** harmonic mean of the two.

In [ ]:
from sklearn.metrics import classification_report
acc = (preds == labels).mean()
baseline = (labels == classes.index('empty')).mean()
print(f'per-square accuracy {acc:.4f}  ({(preds != labels).sum()} wrong of {len(labels)})')
print(f'baseline "always empty" {baseline:.4f}')
print()
print(classification_report(labels, preds, target_names=classes, digits=3))

## 8. Confusion matrix
Rows are the true class, columns the predicted class. The diagonal is correct; every off-diagonal number is a specific mix-up (e.g. bishop read as pawn). Figure for the report.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
fig, ax = plt.subplots(figsize=(10, 9))
ConfusionMatrixDisplay.from_predictions(labels, preds, labels=range(len(classes)), display_labels=classes,
                                        xticks_rotation=60, colorbar=False, values_format='d', ax=ax)
ax.set_title('Test confusion matrix (rows = true, columns = predicted)')
plt.tight_layout(); plt.savefig(f'{DRIVE}/outputs/confusion_matrix.png', dpi=150); plt.show()

## 9. Whole-board accuracy
What a user actually experiences: a photo is only right if all 64 squares are right, since one wrong square gives a wrong FEN and a wrong engine move. Crop file names are `<game>_<photo>_<square>.jpg`, so squares are grouped back into photos.

In [ ]:
import pandas as pd
df = pd.DataFrame({'path': paths, 'true': [classes[i] for i in labels], 'pred': [classes[i] for i in preds]})
df['photo'] = df.path.map(lambda p: '_'.join(Path(p).stem.split('_')[:2]))   # e.g. test01_0001
df['game'] = df.photo.str.split('_').str[0]
df['wrong'] = df.true != df.pred
per_photo = df.groupby('photo').agg(game=('game', 'first'), wrong=('wrong', 'sum'), squares=('wrong', 'size'))
assert (per_photo.squares == 64).all(), 'every photo should have 64 squares'
print(per_photo.groupby('game').agg(photos=('wrong', 'size'),
                                    boards_all_correct=('wrong', lambda w: (w == 0).mean()),
                                    mean_wrong_squares=('wrong', 'mean')))
print()
print(f'overall whole-board accuracy {(per_photo.wrong == 0).mean():.4f}  ({(per_photo.wrong == 0).sum()} of {len(per_photo)} photos)')
print()
print('photos by number of wrong squares:')
print(per_photo.wrong.value_counts().sort_index().rename('photos'))

## 10. Gallery of mistakes
Every misclassified test crop with its true and predicted label (first 24 shown; all are saved to `test_mistakes.csv`). Looking at the actual images explains *why* the model failed (occlusion, a piece leaning into the crop, lighting) for the report's limitations section.

In [ ]:
from PIL import Image
wrong = df[df.wrong]
wrong.drop(columns=['path', 'wrong']).assign(file=wrong.path.map(lambda p: Path(p).name)).to_csv(f'{DRIVE}/outputs/test_mistakes.csv', index=False)
n = min(len(wrong), 24)
if n == 0:
    print('no mistakes on the test set')
else:
    cols = 6
    fig, axes = plt.subplots(-(-n // cols), cols, figsize=(cols * 2.2, -(-n // cols) * 2.7), squeeze=False)
    for ax in axes.flat:
        ax.axis('off')
    for ax, r in zip(axes.flat, wrong.head(n).itertuples()):
        ax.imshow(Image.open(r.path))
        ax.set_title(f'{Path(r.path).stem}\ntrue {r.true}\npred {r.pred}', fontsize=7)
    plt.tight_layout(); plt.savefig(f'{DRIVE}/outputs/test_mistakes.png', dpi=150); plt.show()
print(len(wrong), 'mistakes total')

## 11. Done
Download from `MyDrive/chess_vision/outputs/` into `chess_vision/outputs/` on the laptop: `model.pt` (needed by the live demo), `training_curves.png`, `confusion_matrix.png`, `test_mistakes.png`, `test_mistakes.csv`.